# Ишка: дообучение Qwen3-4B в Google Colab

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → T4 GPU*. Затем запускай ячейки по порядку (Shift+Enter).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код проекта
Скачивает проект из GitHub (или подтягивает обновления, если он уже скачан). Ячейку можно запускать повторно.
Если загрузил `ishka.zip` через панель *Файлы*, ячейка распакует его вместо GitHub.

Сам ноутбук `git pull` не обновляет: после изменений на GitHub открой его в Colab заново.

In [ ]:
import os
if not os.path.exists('/content/project'):
    if os.path.exists('/content/ishka.zip'):  # загрузил архив через панель «Файлы»
        !unzip -q /content/ishka.zip -d /content/project
    else:
        !git clone https://github.com/GO-igor/ishka-assistant.git /content/project
%cd /content/project
if os.path.isdir('.git'):
    !git pull --ff-only || echo "⚠️ git pull не прошёл: обновления не скачались, напиши мне"

## 2. Библиотеки

In [ ]:
!pip install -q unsloth openai

## 3. Датасет: собрать и проверить

In [ ]:
!python db/build_db.py && python gen_dataset.py
!python validate_dataset.py data/train.jsonl | tail -2
!python validate_dataset.py data/eval.jsonl | tail -2

## 4. Точка отсчёта: базовая модель без дообучения (можно пропустить)
Первые 60 примеров, чтобы было быстрее: общие проценты можно сравнить с шагом 6, разбивку по инструментам — нет.

In [ ]:
!python evaluate.py --model unsloth/Qwen3-4B-Instruct-2507 --limit 60

## 5. Обучение
Прогресс: строка вида `12/97 [48:00<5:10:00, ...]` — пройдено шагов из всех, прошло времени и сколько осталось. `loss` должен уменьшаться.

Каждые 10 шагов чекпоинт сохраняется на Google Диск в папку `ishka/ckpt-...`. Когда Colab спросит доступ к Диску, выбери **тот же аккаунт** и отметь **все галочки** разрешений: без этого будет `mount failed`. Если Диск всё же не подключится, обучение пойдёт без чекпоинтов на Диске (ячейка об этом напишет).

**Если Colab отключился** — подключись снова и запусти ячейки 1, 2, 3 и эту: в начале будет строка «Продолжаю с ...checkpoint-N». Это работает и после обновлений на GitHub, если они не меняли данные и настройки обучения. Если отключился уже после обучения — то же самое: ячейка сразу сохранит готовую модель в `out/lora`.

Старые папки `ishka/ckpt-...` на Диске (от прошлых настроек или данных) можно удалить, текущую не трогай.

In [ ]:
import os
from google.colab import drive
os.environ.pop('CKPT_DIR', None)  # от старой версии ноутбука
os.environ.pop('CKPT_ROOT', None)
for attempt in (1, 2):
    try:
        drive.mount('/content/drive', force_remount=attempt == 2)
        os.environ['CKPT_ROOT'] = '/content/drive/MyDrive/ishka'
        break
    except Exception as e:
        print(f'⚠️ Google Диск не подключился ({type(e).__name__}: {e})')
if 'CKPT_ROOT' not in os.environ:
    print('Обучение пойдёт с чекпоинтами только в Colab: если он отключится, начнётся заново.',
          'Чтобы исправить, останови ячейку (■), запусти её снова и в окне доступа',
          'выбери тот же аккаунт и отметь все галочки.')
# Быстрее в ~2 раза, но модель слабее: 'unsloth/Qwen3-1.7B'
os.environ['BASE'] = 'unsloth/Qwen3-4B-Instruct-2507'
!python train_qlora.py

## 6. Оценка дообученной модели
Первая строка — первый ответ на вопрос (какой инструмент), вторая — ответ после результата инструмента: там модель должна ответить текстом, а не вызвать ещё один инструмент.

In [ ]:
!python evaluate.py --model out/lora
!python evaluate.py --model out/lora --step 2

## 7. Сохранить LoRA на Google Диск (чтобы не потерять при отключении Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ishka && cp -r out/lora /content/drive/MyDrive/ishka/ && echo "LoRA на Диске: MyDrive/ishka/lora"

## 8. Экспорт в GGUF (для запуска на ноутбуке/сервере через llama.cpp или Ollama)
Получится `out/gguf/*.gguf` (~2.5 ГБ) и `Modelfile` для Ollama (в нём уже шаблон с вызовом инструментов).

In [ ]:
!python export.py --no-merged
!ls -lh out/gguf

### Скопировать GGUF и Modelfile на Google Диск

In [ ]:
# копия GGUF и Modelfile на Диск, оттуда скачаешь на компьютер
import subprocess
from google.colab import drive
drive.mount('/content/drive')
r = subprocess.run('mkdir -p /content/drive/MyDrive/ishka && cp out/gguf/* /content/drive/MyDrive/ishka/ '
                   '&& ls -lh /content/drive/MyDrive/ishka', shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
drive.flush_and_unmount()  # дождаться, пока файлы реально загрузятся на Диск
print('Готово: файлы на Диске в MyDrive/ishka' if r.returncode == 0
      else '⚠️ Не скопировалось: сначала запусти экспорт (ячейка выше)')

## 9. Загрузить на Hugging Face (по желанию)
Слева *Секреты* (ключ) → добавь `HF_TOKEN` (токен с правом write с huggingface.co/settings/tokens), впиши свой ник в `REPO` и запусти ячейку. Она заново экспортирует модель и загрузит LoRA и GGUF в приватные репозитории.

In [ ]:
import os
from google.colab import userdata
REPO = 'ВАШ_НИК/ishka'  # впиши свой ник на Hugging Face
assert 'ВАШ_НИК' not in REPO, 'Впиши свой ник в REPO'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!python export.py --no-merged --hf {REPO}

## 10. Если на Mac Ишка отвечает странно: проверить LoRA с Диска
Обучать заново не нужно. Сначала запусти ячейки разделов 1 и 2 (код и библиотеки), затем эту.
Первая строка оценки — первый ответ на вопрос, вторая — ответ после результата инструмента (там нужен текст).
Пришли весь вывод: в начале будет список файлов на Диске с датами.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -l --time-style=long-iso /content/drive/MyDrive/ishka /content/drive/MyDrive/ishka/lora
!python evaluate.py --model /content/drive/MyDrive/ishka/lora --limit 60
!python evaluate.py --model /content/drive/MyDrive/ishka/lora --step 2 --limit 60

### Переэкспорт в GGUF из LoRA на Диске
Нужен, если `diag_server.py` на Mac пишет в «Итоге» ✗, а здесь на обоих шагах ≥ 90%: значит, точность потерялась при сжатии в q4_k_m.
Получится `.gguf` в q8_0 (~4.3 ГБ, точнее q4_k_m). Потом запусти ячейку «Скопировать GGUF и Modelfile на Google Диск» (раздел 8).

In [ ]:
!python export.py --lora /content/drive/MyDrive/ishka/lora --quant q8_0 --no-merged
!ls -lh out/gguf